# 05 · Итоговая разметка — в граф

**Зачем этот шаг.** Всё, что мы узнали о технологиях, пока лежит в файлах. Чтобы
этим пользовались поиск, интерфейс и аналитики, результат записывается прямо в
узлы `Technology` графа Neo4j. Тогда в Neo4j Browser можно одним запросом найти
вероятные слабые сигналы или отсеять шум.

**Что записывается** (код: `src/lctrend/modeling/labeling/graph_labels.py`):

| Свойство узла | Откуда | Что значит |
|---|---|---|
| `signal_probability` | модель (04) | вероятность слабого сигнала на последнюю дату |
| `signal_flag` | модель | вероятность выше порога, подобранного на valid |
| `signal_model`, `signal_snapshot` | модель | какая модель и на какую дату оценила |
| `llm_verdict` | LLM (02) | success / niche / faded / junk / mainstream / unclear |
| `llm_is_technology` | LLM | `false` — шум, не технология |
| `llm_score`, `llm_hype`, `llm_maturity` | LLM | последний год: оценка (0 — слабый сигнал), перегретость, сформированность |
| `llm_rationale`, `llm_years` | LLM | обоснование и оценки по всем годам (JSON) |

**Безопасность.** Меняются только свойства с префиксами `signal_` и `llm_` у
существующих узлов: ничего не создаётся и не удаляется. Запись и отмена
выполняются только явными переключателями `WRITE` и `CLEAR`.

**Новые технологии.** Технологии, появившиеся в графе после выгрузки, получат
вероятность модели после новой выгрузки; метки LLM для них можно поставить
сразу: `python -m lctrend.modeling.dataset.llm_outcomes --from-graph --output <run>/dataset/llm_labels.csv`
(продолжит журнал и возьмёт только новые).

**Где мы в цепочке**

1. **01 · выгрузка** — что есть в графе на каждую дату (признаки и окружение технологий);
2. **02 · разметка** — что LLM считает слабым сигналом в каждый год;
3. **03 · выборка** — соединяем 01 и 02 в обучающую таблицу;
4. **04 · обучение** — CatBoost, HGT и их комбинация, метрики;
5. **05 · граф** — записываем итоговую разметку в узлы технологий.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"))
from lab import *
from IPython.display import display, Markdown

from lctrend.modeling.labeling.graph_labels import clear_labels, graph_rows, read_answers, read_scores, write_labels

if not (LABELING_DIR / "scores.csv").exists():
    raise FileNotFoundError(
        "Нет итоговой разметки labeling/scores.csv: сначала выполните 04_training.ipynb — он обучает модели и размечает технологии."
    )
scores = read_scores(LABELING_DIR / "scores.csv")
answers = read_answers(DATASET_DIR / "llm_labels.csv.jsonl")
rows = graph_rows(scores, answers)
props = pd.DataFrame([{"id": row["id"], **row["props"]} for row in rows])
overview = pd.DataFrame([
    ("Технологий к записи", len(rows)),
    ("С вероятностью модели", int(props.signal_probability.notna().sum())),
    ("С разметкой LLM", int(props.llm_verdict.notna().sum())),
    ("Шум (не технология)", int((props.llm_is_technology == False).sum())),
    ("Сигнал по модели (signal_flag)", int(props.signal_flag.fillna(False).sum())),
], columns=["Показатель", "Значение"]).set_index("Показатель")
display(overview.style.format(num))

,Значение
Показатель,
Технологий к записи,3 108
С вероятностью модели,2 991
С разметкой LLM,3 106
Шум (не технология),1 188
Сигнал по модели (signal_flag),119


## 1. Что будет записано: самые вероятные слабые сигналы

**Как читать.** Технологии отсортированы по вероятности модели, шум исключён.
Рядом — мнение LLM: если модель и LLM согласны, сигналу больше доверия.

In [2]:
names = {row["technology_id"]: row["technology"] for row in scores}
view = props.assign(technology=props.id.map(names))
view = view[view.llm_is_technology != False]
display(view.sort_values("signal_probability", ascending=False).head(20)[
    ["technology", "signal_probability", "signal_flag", "llm_verdict", "llm_score", "llm_hype", "llm_maturity"]]
    .rename(columns={"technology": "технология", "signal_probability": "вероятность модели", "signal_flag": "сигнал",
                     "llm_verdict": "вердикт LLM", "llm_score": "оценка LLM", "llm_hype": "перегретость",
                     "llm_maturity": "сформированность"}).round(3).style.hide(axis="index"))

технология,вероятность модели,сигнал,вердикт LLM,оценка LLM,перегретость,сформированность
ACE (Abstract-Concrete-Execute) Security Architecture,0.925000,True,unclear,0.300000,0.200000,0.200000
PropertyGPT,0.879000,True,unclear,0.600000,0.400000,0.200000
Lattice-based Attribute-Based Encryption (ABE),0.868000,True,niche,0.700000,0.200000,0.400000
Hybrid cybersecurity framework for IoE integrating FL and LLMs,0.853000,True,unclear,0.000000,0.200000,0.100000
Confucius framework,0.847000,True,unclear,0.500000,0.300000,0.200000
Agent Laboratory,0.818000,True,unclear,0.400000,0.500000,0.300000
EukPhylo (EukPhylogenomics),0.804000,True,unclear,0.400000,0.100000,0.250000
AI-driven exploit generation,0.802000,True,success,0.650000,0.700000,0.450000
AgentHarm,0.798000,True,niche,0.450000,0.250000,0.400000
Hybrid RIS-aided Digital AirComp (HRD-AirComp),0.778000,True,unclear,0.350000,0.200000,0.100000


## 2. Согласие модели и LLM

**Как читать.** Строки — решение модели, столбцы — мнение LLM о последнем
годе. Диагональ — согласие. Технологии, где модель видит сигнал, а LLM нет (и
наоборот), — первые кандидаты для ручной проверки.

In [3]:
agreement = pd.crosstab(
    view.signal_flag.fillna(False).map({True: "модель: сигнал", False: "модель: не сигнал"}),
    view.llm_score.le(CONFIG["labels"]["weak_max"]).map({True: "LLM: слабый сигнал", False: "LLM: не слабый"}),
)
display(agreement)

llm_score,LLM: не слабый,LLM: слабый сигнал
signal_flag,,
модель: не сигнал,1744,114
модель: сигнал,52,10


## 3. Записать в граф

Запись идёт в общую базу Neo4j. Поставьте `WRITE = True` и выполните ячейку.
Повторная запись просто обновит значения.

In [4]:
from lctrend.session import with_graph

WRITE = False
if WRITE:
    result = await with_graph(lambda store: write_labels(store, rows))
    display(Markdown(
        f"Записано узлов: **{num(result['written'])}**. Не найдено в графе (слиты или удалены): "
        f"{num(len(result['not_in_graph']))}. Сейчас в графе: {result['graph']}."
    ))
else:
    display(Markdown("Запись выключена: `WRITE = False`."))

Запись выключена: `WRITE = False`.

## 4. Как смотреть результат в Neo4j Browser

```cypher
// вероятные слабые сигналы по модели, без шума
MATCH (t:Technology) WHERE t.signal_flag AND t.llm_is_technology
RETURN t.preferred_label, t.signal_probability, t.llm_verdict, t.llm_rationale
ORDER BY t.signal_probability DESC LIMIT 25;

// шум: названия, которые LLM не считает технологиями
MATCH (t:Technology) WHERE t.llm_is_technology = false
RETURN t.preferred_label, t.llm_rationale LIMIT 50;

// расхождения модели и LLM — кандидаты для ручной проверки
MATCH (t:Technology) WHERE t.signal_flag AND t.llm_score >= 0.7
RETURN t.preferred_label, t.signal_probability, t.llm_score, t.llm_rationale;
```

## 5. Отменить запись

Снимает все свойства `signal_*` и `llm_*` с узлов технологий; остальные данные
графа не затрагиваются. Поставьте `CLEAR = True` и выполните.

In [5]:
CLEAR = False
if CLEAR:
    display(Markdown(f"Снято свойств с узлов: **{num(await with_graph(clear_labels))}**."))
else:
    display(Markdown("Очистка выключена: `CLEAR = False`."))

Очистка выключена: `CLEAR = False`.